In [ ]:
# Import TensorFlow for deep learning operations
import tensorflow as tf

# Check whether a GPU is available in the current environment.
# Using a GPU can significantly speed up CNN training.
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# Import TensorFlow for creating and training deep learning models
import tensorflow as tf

# Import NumPy for numerical and array operations
import numpy as np

# Import Matplotlib for data visualization
import matplotlib.pyplot as plt

# Import Pandas for creating and displaying result tables
import pandas as pd

# Import the CIFAR-10 dataset from Keras
from tensorflow.keras.datasets import cifar10

# Import the function used to convert class labels into one-hot encoded vectors
from tensorflow.keras.utils import to_categorical

In [ ]:
# Load the CIFAR-10 dataset.
# The dataset contains 60,000 color images belonging to 10 different classes.
# 50,000 images are used for training and 10,000 images are used for testing.
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Display the shape of the training and testing image datasets
print(x_train.shape)
print(x_test.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 1898s 11us/step
(50000, 32, 32, 3)
(10000, 32, 32, 3)


In [ ]:
# Convert the integer class labels into one-hot encoded vectors.
# There are 10 classes in the CIFAR-10 dataset.
# This format is required because the models use categorical crossentropy loss.
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

In [ ]:
# Define the image size required by the pretrained VGG16 and EfficientNetB0 models
IMG_SIZE = 224

# Define the number of images processed in each training batch
BATCH_SIZE = 32

# Create a TensorFlow Dataset from the training images and labels
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))

# Create a TensorFlow Dataset from the testing images and labels
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))

# Shuffle the training dataset to reduce the effect of the original data order
train_ds = train_ds.shuffle(10000)

# Resize the CIFAR-10 images from 32x32 to 224x224 pixels.
# The images are also converted to float32 before resizing.
train_ds = train_ds.map(
    lambda x, y: (
        tf.image.resize(tf.cast(x, tf.float32), (IMG_SIZE, IMG_SIZE)),
        y
    ),
    num_parallel_calls=tf.data.AUTOTUNE
)

# Apply the same resizing and type conversion to the test dataset
test_ds = test_ds.map(
    lambda x, y: (
        tf.image.resize(tf.cast(x, tf.float32), (IMG_SIZE, IMG_SIZE)),
        y
    ),
    num_parallel_calls=tf.data.AUTOTUNE
)

# Divide the datasets into batches and prefetch data
# to improve the efficiency of the training pipeline.
train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
# Import the pretrained VGG16 architecture
from tensorflow.keras.applications import VGG16

# Import the layers required to create the new classification head
from tensorflow.keras.layers import Input, Dense, Dropout, GlobalAveragePooling2D

# Import Model for creating the complete neural network
from tensorflow.keras.models import Model

# Load VGG16 with pretrained ImageNet weights.
# include_top=False removes the original ImageNet classification layer
# because we are creating a new classifier for CIFAR-10.
vgg_base = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze the pretrained VGG16 layers.
# Their weights will not be updated during training.
vgg_base.trainable = False

# Define the input shape for the model
inputs = Input(shape=(224, 224, 3))

# Pass the input images through the frozen VGG16 feature extractor
x = vgg_base(inputs, training=False)

# Convert the extracted feature maps into a single feature vector
x = GlobalAveragePooling2D()(x)

# Add a fully connected layer for learning CIFAR-10-specific features
x = Dense(256, activation='relu')(x)

# Drop 50% of the neurons randomly during training
# to help reduce overfitting
x = Dropout(0.5)(x)

# Add the final classification layer with 10 output neurons,
# one for each CIFAR-10 class
outputs = Dense(10, activation='softmax')(x)

# Create the complete VGG16-based classification model
vgg_model = Model(inputs, outputs)

# Compile the model using Adam optimizer and categorical crossentropy loss
vgg_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


In [ ]:
# Train the VGG16-based model for 3 epochs.
# The test dataset is used here as validation data to observe performance
# after each training epoch.
history_vgg = vgg_model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=3
)

Epoch 1/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 331s 212ms/step - accuracy: 0.7672 - loss: 0.6772 - val_accuracy: 0.7979 - val_loss: 0.5870
Epoch 2/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 330s 211ms/step - accuracy: 0.7783 - loss: 0.6400 - val_accuracy: 0.8054 - val_loss: 0.5646
Epoch 3/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 330s 211ms/step - accuracy: 0.7871 - loss: 0.6205 - val_accuracy: 0.8103 - val_loss: 0.5528


In [ ]:
# Import the pretrained EfficientNetB0 architecture
from tensorflow.keras.applications import EfficientNetB0

# Load EfficientNetB0 with pretrained ImageNet weights.
# The original ImageNet classification layer is removed.
efficient_base = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze the pretrained EfficientNetB0 layers
# so that only the new classification layers are trained.
efficient_base.trainable = False

# Define the input layer for 224x224 RGB images
inputs = Input(shape=(224, 224, 3))

# Extract image features using the frozen EfficientNetB0 network
x = efficient_base(inputs, training=False)

# Reduce the spatial feature maps to a single feature vector
x = GlobalAveragePooling2D()(x)

# Add a fully connected layer for CIFAR-10 classification
x = Dense(256, activation='relu')(x)

# Apply dropout to reduce the possibility of overfitting
x = Dropout(0.5)(x)

# Create the final output layer with 10 class probabilities
outputs = Dense(10, activation='softmax')(x)

# Create the complete EfficientNetB0-based model
efficient_model = Model(inputs, outputs)

# Compile the model using Adam optimizer,
# categorical crossentropy loss, and accuracy as the evaluation metric
efficient_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [ ]:
# Train the EfficientNetB0-based model for 3 epochs.
# The test dataset is used as validation data during training.
history_efficient = efficient_model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=3
)

Epoch 1/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 121s 59ms/step - accuracy: 0.8460 - loss: 0.4597 - val_accuracy: 0.8957 - val_loss: 0.3102
Epoch 2/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 65s 41ms/step - accuracy: 0.8791 - loss: 0.3514 - val_accuracy: 0.9047 - val_loss: 0.2832
Epoch 3/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 64s 41ms/step - accuracy: 0.8899 - loss: 0.3217 - val_accuracy: 0.9051 - val_loss: 0.2813


In [ ]:
# Evaluate the VGG16 model on the test dataset
vgg_loss, vgg_accuracy = vgg_model.evaluate(test_ds)

# Evaluate the EfficientNetB0 model on the test dataset
efficient_loss, efficient_accuracy = efficient_model.evaluate(test_ds)

# Display the test accuracy of the VGG16 model as a percentage
print("VGG16 Accuracy:", vgg_accuracy * 100)

# Display the test accuracy of the EfficientNetB0 model as a percentage
print("EfficientNetB0 Accuracy:", efficient_accuracy * 100)

313/313 ━━━━━━━━━━━━━━━━━━━━ 56s 180ms/step - accuracy: 0.8103 - loss: 0.5528
313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 34ms/step - accuracy: 0.9051 - loss: 0.2813
VGG16 Accuracy: 81.02999925613403
EfficientNetB0 Accuracy: 90.50999879837036


In [ ]:
# Create a Pandas DataFrame to compare the accuracy
# of the two transfer-learning models
results = pd.DataFrame({
    'Model': ['VGG16', 'EfficientNetB0'],

    # Convert the accuracy values from decimal form to percentage
    'Accuracy (%)': [
        vgg_accuracy * 100,
        efficient_accuracy * 100
    ]
})

# Display the comparison table
results

,Model,Accuracy (%)
0,VGG16,81.029999
1,EfficientNetB0,90.509999
